# Lab 14: The Simplicity and Power of Non-Linearity

**Computer Vision Course**

This lab explores **non-linear operations** in image processing — techniques that don't follow the simple linear rules but are incredibly powerful!

**What you'll learn:**
- Histogram equalization for contrast enhancement
- Median filtering for noise removal (non-linear!)
- Adaptive thresholding for segmentation
- Why non-linearity matters in computer vision

**What you'll build:**
- A histogram equalization algorithm from scratch
- A median filter implementation
- An adaptive thresholding system

**Why this matters:**
Non-linear operations solve problems linear methods cannot:
- Linear smoothing blurs edges → Median preserves edges!
- Global threshold fails on varied lighting → Adaptive succeeds!
- Histogram tells us pixel distribution → Equalization improves it!

**Connection to previous labs:**
- Lab 7: Linear smoothing (Gaussian) → Today: Non-linear smoothing (median)
- Lab 8-9: Linear filters → Today: Non-linear transformations
- Looking ahead: Neural networks are highly non-linear!

## Setup

In [1]:
"""
Computer Vision Course - Lab 14: Non-Linearity

This cell sets up the environment.
Works automatically for both local and Google Colab!
"""

import os
import sys

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

print("=" * 60)
print("Computer Vision - Lab 14: Non-Linearity")
print("=" * 60)

if IN_COLAB:
    print("\n🔵 Running on Google Colab")
    print("-" * 60)

    if not os.path.exists('computer-vision'):
        print("📥 Cloning repository...")
        !git clone https://github.com/mjck/computer-vision.git
        print("✓ Repository cloned successfully")
    else:
        print("✓ Repository already exists")
        !git -C computer-vision pull origin main
        print("✓ Repository updated successfully")

    %cd computer-vision/labs/lab14_nonlinearity
    print(f"✓ Current directory: {os.getcwd()}")

    sys.path.insert(0, '/content/computer-vision')
    print("✓ Python path configured")

    print("-" * 60)
    print("🟢 Colab setup complete!\n")

else:
    print("\n🟢 Running locally")
    print("-" * 60)
    print(f"✓ Current directory: {os.getcwd()}")

    repo_root = os.path.abspath('../..')
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print(f"✓ Repository root: {repo_root}")

    print("-" * 60)
    print("🟢 Local setup complete!\n")

print("=" * 60)
print("✅ Environment ready!")
print("=" * 60)

Computer Vision - Lab 13: Non-Linearity

🔵 Running on Google Colab
------------------------------------------------------------
📥 Cloning repository...
Cloning into 'computer-vision'...
remote: Enumerating objects: 506, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 506 (delta 18), reused 49 (delta 11), pack-reused 446 (from 1)
Receiving objects: 100% (506/506), 9.23 MiB | 22.49 MiB/s, done.
Resolving deltas: 100% (236/236), done.
Filtering content: 100% (144/144), 39.74 MiB | 17.29 MiB/s, done.
✓ Repository cloned successfully
[Errno 2] No such file or directory: 'computer-vision/labs/lab13_nonlinearity'
/content
✓ Current directory: /content
✓ Python path configured
------------------------------------------------------------
🟢 Colab setup complete!

✅ Environment ready!


## Import Libraries

In [ ]:
import numpy as np
import seaborn as sns
import cv2
from matplotlib import pyplot as plt

# Import course utilities
try:
    from sdx import cv_imread, cv_imshow, cv_grayread
    print("✓ sdx module loaded")
except ImportError as e:
    print(f"❌ Could not import sdx: {e}")
    raise

# Configure matplotlib and seaborn
plt.rcParams['figure.figsize'] = [12.8, 4.8]
sns.set_theme()

print("✓ All imports successful")

---
## Part 1: Histogram Equalization

**The problem:** Many images have poor contrast — all pixels clustered in a narrow range.

**The solution:** Spread pixel values across the full 0-255 range!

**Why non-linear?** Histogram equalization is a non-linear transformation — different input values can have different amounts of "stretching".

### How Histogram Equalization Works

1. **Compute histogram:** Count pixels at each intensity level
2. **Compute cumulative histogram:** Running sum of histogram
3. **Map old intensities to new:** Use cumulative distribution function (CDF)

**The mapping:** `new_value = 255 × CDF(old_value)`

This spreads out the most common values and compresses rare ones.

### Convenience Functions

Helper functions for computing and visualizing histograms.

In [ ]:
def histogram(image):
    """
    Compute normalized histogram of grayscale image.

    Args:
        image: Grayscale image (uint8)

    Returns:
        Array of 256 values (probability for each intensity level)
    """
    values = image.flatten()
    bins = range(0, 256)
    counts, _ = np.histogram(values, bins, density=True)
    return counts

print("✓ histogram() function defined")

In [ ]:
def plot_histogram(counts):
    """
    Visualize histogram using seaborn barplot.

    Args:
        counts: Array of histogram values
    """
    p = sns.barplot(x=range(0, 255), y=counts)

    p.set_ylim(-0.02, 1.015)
    p.set_yticks([])

    p.set_xlim(-1.75, 256)
    p.set_xticks([])

    p.plot()

print("✓ plot_histogram() function defined")

### Load Test Image

We'll use a landscape photo with poor contrast.

In [ ]:
# Load landscape image
landscape = cv_grayread('landscape.png')

print(f"Image shape: {landscape.shape}")
print(f"Intensity range: [{landscape.min()}, {landscape.max()}]")

cv_imshow(landscape)
print("Notice: Image looks washed out, lacks contrast")

### Visualize Original Histogram

In [ ]:
# Compute and plot histogram
h = histogram(landscape)

print("Original histogram:")
plot_histogram(h)
print("\nNotice: Pixels concentrated in narrow range (low contrast)")

---
## Activity 1a: Cumulative Histogram

**Task:** Implement a function that computes the cumulative histogram.

**Definition:** For cumulative histogram `ch`:
```
ch(l) = Σ h(k) for all k ≤ l
```

In other words, `ch[i]` is the sum of all histogram values from 0 to i.

**Requirements:**
- Input: `counts` array (histogram)
- Output: `cumulative` array (same size)
- **Avoid loops!** Use NumPy operations

**Hint:** Look up `np.cumsum()` — it does exactly this!

**What it represents:** `ch[L]` = fraction of pixels with intensity ≤ L

In [ ]:
# ── Activity 1a: Your code here ───────────────────────────────────────────

def cumulative(counts):
    """
    Compute cumulative histogram.

    Args:
        counts: Histogram array

    Returns:
        Cumulative histogram array
    """
    return counts.copy()  # TODO: Replace with your implementation

# ──────────────────────────────────────────────────────────────────────────

print("Implement cumulative() function above")

### Test Cumulative Histogram

In [ ]:
# Compute and plot cumulative histogram
ch = cumulative(h)

print("Cumulative histogram:")
plot_histogram(ch)
print("\nShould show cumulative distribution (monotonically increasing)")
print("Final value should be close to 1.0 (100% of pixels)")

---
## Activity 1b: Histogram Equalization

**Task:** Implement histogram equalization from scratch.

**The algorithm:**
```
For each pixel with intensity old_level:
    1. Look up ch[old_level] (cumulative probability)
    2. new_level = ch[old_level] × 255
    3. Set pixel to new_level
```

**The math:** This maps:
- `ch[old_level] = 0%` → `new_level = 0`
- `ch[old_level] = 50%` → `new_level = 127`
- `ch[old_level] = 100%` → `new_level = 255`

**Linear relationship:** `new_level = 255 × ch[old_level]`

**Requirements:**
- Use the provided loop structure
- Replace the `new_level = old_level` line
- Make sure values are in [0, 255] range

**Hint:** It really is as simple as `255 * ch[old_level]`!

In [ ]:
# ── Activity 1b: Your code here ───────────────────────────────────────────

height, width = landscape.shape

equalized = np.empty((height, width))

for y in range(height):
    for x in range(width):
        old_level = landscape[y, x]

        # TODO: Replace this line with correct formula
        new_level = old_level  # Replace this line with your code

        equalized[y, x] = new_level

# ──────────────────────────────────────────────────────────────────────────

cv_imshow(equalized)
print("\nEqualized image should have better contrast!")

### Examine Equalized Histograms

Let's verify that equalization worked:

In [ ]:
# Histogram of equalized image
eh = histogram(equalized)

print("Equalized histogram:")
plot_histogram(eh)
print("\nShould be more spread out than original")

In [ ]:
# Cumulative histogram of equalized image
ceh = cumulative(eh)

print("Cumulative histogram of equalized image:")
plot_histogram(ceh)
print("\nShould be approximately linear (uniform distribution)")
print("This is the goal of equalization!")

---
## Part 2: Median Smoothing

**The problem:** Gaussian smoothing (Lab 7) blurs edges AND removes noise.

**The question:** Can we remove noise while preserving edges?

### Why Median is Non-Linear

**Linear filter (Gaussian):**
```
output[i] = Σ weights[j] × input[i+j]
```
Weighted average — linear combination.

**Non-linear filter (Median):**
```
output[i] = median(input[i-r:i+r])
```
Select middle value — NOT a linear combination!

### Why Median Preserves Edges

**At an edge:**
- Window contains: [10, 10, 10, 200, 200, 200]
- Mean: (10+10+10+200+200+200)/6 = 105 (blurred!)
- Median: Middle values → 10 or 200 (sharp edge!)

**With noise:**
- Window contains: [50, 51, 250 (noise!), 52, 49, 51]
- Mean: (50+51+250+52+49+51)/6 = 84 (noise affects result)
- Median: 51 (noise ignored!)

### Image Selection

We have **6 sources** and **4 noise levels** = 24 test images!

Test different combinations to see how median filtering performs.

### Choose Source Image

Options: `atletica`, `consulting`, `smash`, `insper`, `harvard`, `informatica`

In [ ]:
SOURCE = 'smash'  # Change to test different images

print(f"✓ Selected source: {SOURCE}")

### Choose Noise Level

Options: `5`, `10`, `25`, `50` (percentage of noise)

In [ ]:
LEVEL = 5  # Change to test different noise levels

print(f"✓ Selected noise level: {LEVEL}%")
print("\nTry different levels: 5 (light), 10 (medium), 25 (heavy), 50 (extreme)")

### Load Noisy Image

In [ ]:
# Load chosen noisy image
noisy = cv_grayread(f'{SOURCE}-{LEVEL}.png')

print(f"Noisy image: {noisy.shape}")
cv_imshow(noisy)
print(f"\n{SOURCE} with {LEVEL}% noise")
print("Try changing SOURCE and LEVEL above to see different images!")

---
## Activity 2: Median Smoothing (4 points)

**Task:** Implement median filtering to remove noise while preserving edges.

**Algorithm:**
```
For each pixel (y, x):
    1. Extract window around pixel
    2. Compute median of window
    3. Set output[y, x] = median
```

**Requirements:**
- Window size `n` (e.g., 3 for 3×3, 5 for 5×5)
- Handle borders (use valid region like Lab 7)
- Replace the line `denoised = noisy.copy()`

**Hints:**
- `np.median()` computes the median
- Extract window: `window = image[y-r:y+r+1, x-r:x+r+1]` where `r = n//2`
- Don't forget to iterate only over valid pixels (not too close to border)

**Challenge:** Try different window sizes (3, 5, 7, 9) — what do you observe?

**Serious request:** Test on multiple SOURCE and LEVEL combinations. You'll be surprised how well median filtering works, even at high noise levels!

In [ ]:
# ── Activity 2: Your code here ────────────────────────────────────────────

n = 3  # Size of the window (try 3, 5, 7, 9)

# TODO: Implement median filtering
# Hint: Loop over valid pixels, extract window, compute median
denoised = noisy.copy()  # Replace this line with your code

# ──────────────────────────────────────────────────────────────────────────

cv_imshow(denoised)
print(f"\nMedian filtered with window size {n}×{n}")
print("\n✓ Try different window sizes and noise levels!")
print("✓ Compare to Gaussian smoothing (Lab 7) — which preserves edges better?")

---
## Part 3: Thresholding Algorithms

**The problem:** Separate foreground from background (segmentation).

**Simple threshold:** `if pixel < T then 0 else 255`

**Challenge:** What if lighting is non-uniform?
- Bright areas: text might be > global threshold
- Dark areas: background might be < global threshold
- Simple threshold fails!

**Solution:** Adaptive thresholding — different threshold for different regions!

### Why This is Non-Linear

**Linear operation:** `output = a × input + b`

**Thresholding:**
```
output = 0   if input < T
output = 255 if input ≥ T
```

This is a **step function** — highly non-linear!

### Load Sudoku Image

This image has text we want to extract.

In [ ]:
# Load sudoku puzzle image
sudoku = cv_grayread('sudoku.png')

print(f"Sudoku image: {sudoku.shape}")
cv_imshow(sudoku)
print("\nGoal: Extract all numbers clearly")

---
## Challenge: Adaptive Thresholding (Bonus +1 point)

**Task:** Implement a thresholding algorithm that preserves all numbers.

**Requirements:**
- **Without using any OpenCV thresholding functions!**
- Feel free to research *ideas* (Otsu's method, adaptive threshold, etc.)
- But you must implement the algorithm yourself

**Why the challenge?**
- A simple global threshold (like `binary = sudoku < 128`) won't work well
- Lighting varies across the image
- Need adaptive or sophisticated approach

**Hints:**
- Try local/adaptive thresholds (different T for different regions)
- Or try Otsu's method (finds optimal global threshold)
- Or try other techniques you research!

**Success criteria:** All numbers clearly visible, minimal background noise

**References:**
- Otsu's method: finds threshold that maximizes between-class variance
- Adaptive threshold: compute local threshold for each region
- Niblack/Sauvola methods: local mean ± k × std

**Note:** OpenCV has `cv2.threshold()` and `cv2.adaptiveThreshold()` — but don't use them for this challenge! Implement from scratch.

In [ ]:
# ── Challenge: Your code here ─────────────────────────────────────────────

# Example of what NOT to do (too simple, won't work well):
binary = np.where(sudoku < 128, 0, 255)

# TODO: Implement your own thresholding algorithm!
# Research ideas: Otsu, adaptive threshold, local methods
# But implement them yourself (no cv2.threshold or cv2.adaptiveThreshold)

# ──────────────────────────────────────────────────────────────────────────

cv_imshow(binary)
print("\nChallenge: Find a threshold that preserves all numbers")
print("Hint: Global threshold (like 128) won't work well here")
print("Research: Otsu's method, adaptive thresholding, Niblack, Sauvola")

---
## Summary

### What You Learned

1. **Histogram Equalization**
   - Non-linear contrast enhancement
   - Maps using cumulative distribution function
   - Spreads pixel values across full range
   - Makes low-contrast images clearer

2. **Median Filtering**
   - Non-linear noise removal
   - Preserves edges (unlike Gaussian)
   - Robust to outliers (salt-and-pepper noise)
   - Choose middle value, not average

3. **Thresholding**
   - Non-linear segmentation
   - Converts grayscale → binary
   - Adaptive methods handle varying illumination
   - Foundation for OCR, document scanning

### Key Insights

**Why non-linearity matters:**
- **Linear methods** (Gaussian, averaging) blur everything
- **Non-linear methods** (median, threshold) can be selective
- Median: removes noise, keeps edges
- Histogram equalization: custom mapping per value
- Thresholding: step function, not linear

**Limitations and trade-offs:**
- Histogram equalization: can amplify noise
- Median filtering: slower than Gaussian, can lose fine details
- Global thresholding: fails on non-uniform illumination
- Adaptive methods: more complex, parameter-sensitive

### Connection to Deep Learning

**Neural networks are highly non-linear!**

**ReLU activation:**
```
f(x) = max(0, x)
```
Non-linear, like thresholding!

**Why DNNs need non-linearity:**
- Stack of linear layers = just one linear layer
- Non-linearity enables complex functions
- Median-like selection happens in max-pooling
- Histogram features used in classic CNN designs

**Today's concepts in DNNs:**
- Median ≈ max-pooling (select, don't average)
- Threshold ≈ ReLU activation
- Histogram ≈ feature distribution normalization

### Real-World Applications

**Histogram equalization:**
- Medical imaging (enhance X-rays, MRIs)
- Underwater photography
- Low-light enhancement
- Satellite imagery

**Median filtering:**
- Photo editing (noise removal)
- Video processing (remove compression artifacts)
- Medical imaging (preserve edges)
- Astronomical images

**Thresholding:**
- OCR (optical character recognition)
- Document scanning
- QR code reading
- Cell counting in microscopy
- License plate recognition